# Pipeline de Ingestão da Tabela de Candidatos

Notebook responsável pela ingestão da tabela **consulta_candidatos** utilizando PySpark e Delta Lake. O fluxo realiza a leitura dos arquivos CSV brutos, consolida os dados em um DataFrame e grava a camada Bronze no formato Delta.

#### 1. Importação das Bibliotecas

Nesta etapa importamos as bibliotecas necessárias para realizar a ingestão da tabela **consulta_candidatos**. Utilizamos o Spark para processamento distribuído e o Delta Lake para persistência dos dados na camada Bronze.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip

print('Bibliotecas importadas com sucesso!')

Bibliotecas importadas com sucesso!


#### 2. Inicialização da Sessão Spark

Inicializamos a sessão Spark configurada para trabalhar com o Delta Lake. Essa sessão será utilizada durante todo o processo de ingestão da tabela de candidatos.

In [2]:
builder = (
    SparkSession.builder
        # .master('local[*]')
        .appName('TSE-Analytics-Validation')
        .config(
            'spark.sql.extensions',
            'io.delta.sql.DeltaSparkSessionExtension'
        )
        .config(
            'spark.sql.catalog.spark_catalog',
            'org.apache.spark.sql.delta.catalog.DeltaCatalog'
        )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print('Sessão Spark criada com sucesso com suporte a Delta Lake!')
print(f'Versão do PySpark: {spark.version}')

:: loading settings :: url = jar:file:/usr/local/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/vscode/.ivy2.5.2/cache
The jars for the packages stored in: /home/vscode/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2010ecec-0693-4d93-8980-2b77bffbba4a;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.3.0 in central
	found io.delta#delta-storage;4.3.0 in central
	found io.unitycatalog#unitycatalog-client;0.5.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.5.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.3.0 in

Sessão Spark criada com sucesso com suporte a Delta Lake!
Versão do PySpark: 4.1.1


#### 3. Leitura dos Arquivos da Tabela de Candidatos e Carga na Camada Raw (Parquet)

Lemos os arquivos CSV brutos de candidatos para as eleições de 2000 a 2024. Aplicamos uma padronização rigorosa nos tipos das colunas identificadoras (como `SQ_CANDIDATO`, `SG_UE` e `NR_CPF_CANDIDATO`) para garantir consistência física entre diferentes partições anuais no formato **Parquet**. Os arquivos limpos são salvos por ano na Camada Raw.

In [ ]:
anos = list(range(2000, 2025, 2))

for ano in anos:

    df = (spark.read
               .format('csv')
               .option('header', 'true')    
               .option('sep', ';')
               .option('encoding', 'ISO-8859-1')
               .option('InferSchema', 'true')
               .load(f'data/{ano}/consulta_cand_*/*BRASIL.csv'))
    
    # Padronização de tipos de colunas para evitar conflitos de tipos no Parquet e erros de conversão de esquema
    if "SQ_CANDIDATO" in df.columns:
        df = df.withColumn("SQ_CANDIDATO", F.col("SQ_CANDIDATO").cast("bigint"))

    if "SQ_COLIGACAO" in df.columns:
        df = df.withColumn("SQ_COLIGACAO", F.col("SQ_COLIGACAO").cast("bigint"))

    if "SG_UE" in df.columns:
        df = df.withColumn("SG_UE", F.col("SG_UE").cast("string"))

    if "NR_CPF_CANDIDATO" in df.columns:
        df = df.withColumn("NR_CPF_CANDIDATO", F.col("NR_CPF_CANDIDATO").cast("string"))

    if "NR_TITULO_ELEITORAL_CANDIDATO" in df.columns:
        df = df.withColumn("NR_TITULO_ELEITORAL_CANDIDATO", F.col("NR_TITULO_ELEITORAL_CANDIDATO").cast("string"))

    if "NR_PROCESSO" in df.columns:
        df = df.withColumn("NR_PROCESSO", F.col("NR_PROCESSO").cast("string"))

    if "NR_PROTOCOLO_CANDIDATURA" in df.columns:
        df = df.withColumn("NR_PROTOCOLO_CANDIDATURA", F.col("NR_PROTOCOLO_CANDIDATURA").cast("string"))

    if "SQ_SUBSTITUIDO" in df.columns:
        df = df.withColumn("SQ_SUBSTITUIDO", F.col("SQ_SUBSTITUIDO").cast("bigint"))

    if "VR_DESPESA_MAX_CAMPANHA" in df.columns:
        df = df.withColumn("VR_DESPESA_MAX_CAMPANHA", F.col("VR_DESPESA_MAX_CAMPANHA").cast("double"))

    df.coalesce(1).write.format('parquet').mode('overwrite').save(f'data/raw/consulta_candidatos/{ano}')

26/07/02 12:39:19 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/2000/consulta_cand_*/*BRASIL.csv.
java.io.FileNotFoundException: File data/2000/consulta_cand_*/*BRASIL.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache

#### 4. Consolidação e Carga na Camada Bronze (Delta lake)

Após a leitura dos arquivos CSV de todos os anos, realizamos a leitura consolidada dos arquivos Parquet gerados e gravamos a tabela **consulta_candidatos** no formato Delta. Essa etapa representa a ingestão da tabela na camada Bronze do Data Lake.

In [ ]:
df = (spark.read
           .format('parquet')
           .option('inferSchema', 'true')
           .load('data/raw/consulta_candidatos/*'))

(df.coalesce(1)
   .write
   .format('delta')
   .mode('overwrite')
   .save('data/bronze/consulta_candidatos'))

26/07/03 20:14:07 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/raw/consulta_candidatos/*.
java.io.FileNotFoundException: File data/raw/consulta_candidatos/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.cat

#### 5. Leitura da Tabela Bronze e Criação da View Temporária

Nesta etapa, carregamos a tabela consulta_candidatos armazenada em formato Delta na camada Bronze para um DataFrame do PySpark. Em seguida, registramos esse DataFrame como uma view temporária chamada candidatos, permitindo a execução de consultas SQL durante as análises e validações do processo de ingestão.

In [6]:
candidatos = (spark.read
                   .format('delta')
                   .load('data/bronze/consulta_candidatos'))

candidatos.createOrReplaceTempView('candidatos')

#### 6. Validação da Ingestão

Por fim, realizamos consultas para validar se a ingestão foi concluída com sucesso, verificando a quantidade de registros por ano eleitoral e disponibilizando a tabela para consultas SQL.

In [ ]:
spark.sql(
    """ 
    SELECT
        ANO_ELEICAO,
        COUNT(*) AS TOTAL_CANDIDATOS
    FROM candidatos

    GROUP BY ANO_ELEICAO
    ORDER BY ANO_ELEICAO
    """
).show()

+-----------+----------------+
|ANO_ELEICAO|TOTAL_CANDIDATOS|
+-----------+----------------+
|       2000|          399455|
|       2002|           18109|
|       2004|          402157|
|       2006|           19303|
|       2008|          382079|
|       2010|           22577|
|       2012|          483741|
|       2014|           26263|
|       2016|          498391|
|       2018|           29287|
|       2020|          558804|
|       2022|           29322|
|       2024|          463833|
+-----------+----------------+



A consulta seleciona as colunas mais relevantes e ordena os registros por ano da eleição e turno, permitindo validar a estrutura, o conteúdo e a consistência dos dados carregados na camada Bronze.

In [ ]:
spark.sql(
    """ 
    SELECT
    ANO_ELEICAO,
    CD_TIPO_ELEICAO,
    NR_TURNO,
    DS_ELEICAO,
    SG_UF,
    SG_UE,
    CD_CARGO,
    DS_CARGO,
    SQ_CANDIDATO,
    NR_CANDIDATO,
    NM_CANDIDATO,
    NM_URNA_CANDIDATO,
    NM_SOCIAL_CANDIDATO,
    NR_CPF_CANDIDATO,
    NM_EMAIL,
    CD_SITUACAO_CANDIDATURA,
    DS_SITUACAO_CANDIDATURA,
    CD_DETALHE_SITUACAO_CAND,
    NR_PARTIDO,
    SG_PARTIDO,
    NM_COLIGACAO,
    DS_COMPOSICAO_COLIGACAO,
    DS_NACIONALIDADE,
    SG_UF_NASCIMENTO,
    CD_MUNICIPIO_NASCIMENTO,
    NM_MUNICIPIO_NASCIMENTO,
    DT_NASCIMENTO,
    NR_IDADE_DATA_POSSE,
    NR_TITULO_ELEITORAL_CANDIDATO,
    DS_GENERO,
    CD_GRAU_INSTRUCAO,
    DS_GRAU_INSTRUCAO,
    DS_ESTADO_CIVIL,
    DS_COR_RACA,
    DS_OCUPACAO,
    DS_SIT_TOT_TURNO,
    ST_DECLARAR_BENS
    FROM candidatos

    ORDER BY ANO_ELEICAO, NR_TURNO
    """
).show()

+-----------+---------------+--------+--------------------+-----+-----+--------+----------+------------+------------+--------------------+-----------------+-------------------+----------------+--------+-----------------------+-----------------------+------------------------+----------+----------+--------------------+-----------------------+----------------+----------------+-----------------------+-----------------------+-------------+-------------------+-----------------------------+---------+-----------------+-----------------+---------------+-----------+--------------------+----------------+----------------+
|ANO_ELEICAO|CD_TIPO_ELEICAO|NR_TURNO|          DS_ELEICAO|SG_UF|SG_UE|CD_CARGO|  DS_CARGO|SQ_CANDIDATO|NR_CANDIDATO|        NM_CANDIDATO|NM_URNA_CANDIDATO|NM_SOCIAL_CANDIDATO|NR_CPF_CANDIDATO|NM_EMAIL|CD_SITUACAO_CANDIDATURA|DS_SITUACAO_CANDIDATURA|CD_DETALHE_SITUACAO_CAND|NR_PARTIDO|SG_PARTIDO|        NM_COLIGACAO|DS_COMPOSICAO_COLIGACAO|DS_NACIONALIDADE|SG_UF_NASCIMENTO|CD_MUN